In [2]:

# Cell 1: Inspect BEFORE cleaning

import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Locate project root automatically
current = Path.cwd()

PROJECT_ROOT = next(
    (p for p in [current, *current.parents] if (p / "config.py").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root containing config.py not found")

# Add project root to Python's module search path
sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
import config as cfg
from src.cleaning import clean_data

# Load raw dataset
raw = pd.read_csv(cfg.RAW_DIR / "placement_truth.csv")

# Inspect dataset structure
raw.info()

# Check missing values
print("\nMissing values:")
print(raw.isna().sum())

# Check duplicate records
print("\nDuplicates:", raw.duplicated().sum())

# Check invalid CGPA values
print("CGPA > 10:", (raw["cgpa"] > 10).sum())


<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   application_id    15000 non-null  int64  
 1   student_id        15000 non-null  int64  
 2   company_id        15000 non-null  str    
 3   tenth_pct         15000 non-null  float64
 4   twelfth_pct       15000 non-null  float64
 5   jee_percentile    15000 non-null  float64
 6   cgpa              15000 non-null  float64
 7   company_category  15000 non-null  str    
 8   eligible          15000 non-null  bool   
 9   logical           2990 non-null   float64
 10  quant             2990 non-null   float64
 11  coding            2990 non-null   float64
 12  english           2990 non-null   float64
 13  behavioral        2990 non-null   float64
 14  oa_total          2990 non-null   float64
 15  oa_cleared        2990 non-null   object 
 16  technical         960 non-null    float64
 17  hr  

In [3]:
# Cell 2: clean and show the log
clean, log = clean_data(raw)
pd.Series(log)

rows_in                                     15000
duplicates_removed                              0
tenth_pct_text_fixed                            0
twelfth_pct_text_fixed                          0
tenth_pct_impossible                            0
twelfth_pct_impossible                          0
cgpa_typos_fixed                                0
tenth_pct_recovered_from_other_rows             0
tenth_pct_still_missing                         0
twelfth_pct_recovered_from_other_rows           0
twelfth_pct_still_missing                       0
jee_percentile_recovered_from_other_rows        0
jee_percentile_still_missing                    0
cgpa_recovered_from_other_rows                  0
cgpa_still_missing                              0
logical_solved_from_total                       0
quant_solved_from_total                         0
coding_solved_from_total                        0
english_solved_from_total                       0
behavioral_solved_from_total                    0


In [4]:
# Cell 3: validate against the answer key (the payoff of keeping placement_truth.csv)
truth = pd.read_csv(cfg.RAW_DIR / "placement_truth.csv").set_index("application_id")
c = clean.set_index("application_id")
t = truth.loc[c.index]

for col in ["tenth_pct", "twelfth_pct", "cgpa", "jee_percentile"]:
    ok = np.isclose(c[col], t[col], atol=0.01) | c[col].isna()
    print(f"{col:15s} wrong: {(~ok).sum():4d} | still missing: {c[col].isna().sum()}")

for s in cfg.OA_SECTIONS:
    both = c[s].notna() & t[s].notna()
    print(f"{s:11s} max error:", (c.loc[both, s] - t.loc[both, s]).abs().max().round(3))
    

tenth_pct       wrong:    0 | still missing: 0
twelfth_pct     wrong:    0 | still missing: 0
cgpa            wrong:    0 | still missing: 0
jee_percentile  wrong:    0 | still missing: 0
logical     max error: 0.0
quant       max error: 0.0
coding      max error: 0.0
english     max error: 0.0
behavioral  max error: 0.0


In [5]:
# Cell 4: final checks
print(clean.shape, "| rows should be 15000:", len(clean) == 15000)
print("selected True:", clean["selected"].sum(), "(should be 96)")

(15000, 21) | rows should be 15000: True
selected True: 96 (should be 96)


In [6]:
print("raw JEE missing:", raw["jee_percentile"].isna().sum())
print("clean JEE missing:", clean["jee_percentile"].isna().sum())
print("jee_missing flag total:", clean["jee_missing"].sum())
print("raw rows:", len(raw), "| duplicates:", raw.duplicated().sum())

raw JEE missing: 0
clean JEE missing: 0
jee_missing flag total: 0
raw rows: 15000 | duplicates: 0


In [7]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: d:\shrey\Documents\ML- PROJECT\.venv312\Scripts\python.exe


In [8]:
print("JEE missing in clean:", clean["jee_percentile"].isna().sum(), "(expect 1265)")
print("jee_missing flag total:", clean["jee_missing"].sum(), "(expect 1265)")

JEE missing in clean: 0 (expect 1265)
jee_missing flag total: 0 (expect 1265)


In [9]:
import pandas as pd, config as cfg
from src.cleaning import clean_data

raw = pd.read_csv(cfg.RAW_DIR / "placement_raw.csv")
print("RAW_DIR:", cfg.RAW_DIR)
print("raw rows:", len(raw), "(expect 15146)")
print("raw JEE missing:", raw["jee_percentile"].isna().sum(), "(expect 1265)")

clean, log = clean_data(raw)
print("clean JEE missing:", clean["jee_percentile"].isna().sum(), "(expect 1265)")
print("jee_missing flag:", clean["jee_missing"].sum())

saved = pd.read_csv(cfg.PROCESSED_DIR / "placement_clean.csv")
print("saved file JEE missing:", saved["jee_percentile"].isna().sum())

RAW_DIR: D:\shrey\Documents\ML- PROJECT\data\raw
raw rows: 15146 (expect 15146)
raw JEE missing: 1276 (expect 1265)
clean JEE missing: 1265 (expect 1265)
jee_missing flag: 1265
saved file JEE missing: 1265
